# Get model predictions to looming stimuli

In [ ]:
import numpy as np
import torch
from nnfabrik.utility.nn_helpers import set_random_seed
seed = 42
set_random_seed(seed)

import sys
sys.path.append(".../looming_stimulus/sensorium_2023")
sys.path.append(".../looming_stimulus/neuralpredictors")
from neuralpredictors.training import device_state
from sensorium.datasets.mouse_video_loaders import mouse_video_loader
from sensorium.models.make_model import make_video_model

modelfile = ".../data_retina_sc_fluor_model_fac3d_seed_111_v4a-rs1_10datasets.pth"

## Prepare data and model

In [ ]:
source_dir = '/data/'
paths = [
         source_dir + 'dynamic28188-16-3-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic28188-16-5-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic28188-17-2-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic28188-18-4-Fluorescence-7b721b-v4a/', 
         source_dir + 'dynamic28188-19-9-Fluorescence-7b721b-v4a/', 
         source_dir + 'dynamic28712-3-8-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic29163-2-7-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic29163-4-4-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic29163-5-8-Fluorescence-7b721b-v4a/',
         source_dir + 'dynamic29163-6-5-Fluorescence-7b721b-v4a/',
         ]

source_neuron_ids_path = "/data-quality/"
neuron_ids_paths = [
                    'dynamic28188-16-3-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic28188-16-5-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic28188-17-2-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic28188-18-4-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic28188-19-9-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic28712-3-8-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic29163-2-7-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic29163-4-4-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic29163-5-8-Fluorescence-7b7_neurons_fluor_good.npy',
                    'dynamic29163-6-5-Fluorescence-7b7_neurons_fluor_good.npy',
                    ]

neuron_ids_paths = [source_neuron_ids_path + _ for _ in neuron_ids_paths]

data_loaders = mouse_video_loader(
    paths=paths,
    neuron_ids=[np.load(temp) for temp in neuron_ids_paths],
    batch_size=1,
    subtract_response_min=True,
    include_behavior=True,
    include_pupil_centers=True,
    cuda=True, 
    behavior_channels=[0,2],
    random_sample_within_snippet_flag=False,
) 

device = torch.device("cuda:0") 
cuda_flag = False if device == torch.device("cpu") else True
print (f'cuda_flag: {cuda_flag}')

datakey = list(data_loaders['train'].keys())[0]

In [ ]:
factorised_3D_core_dict = dict(
    input_channels=3,
    hidden_channels=[64, 64, 64],
    spatial_input_kernel=(11,11),
    temporal_input_kernel=11,
    spatial_hidden_kernel=(5,5),
    temporal_hidden_kernel=5,
    stride=1,
    layers=3,
    gamma_input_spatial=10,
    gamma_input_temporal=0.01, 
    bias=True, 
    hidden_nonlinearities='elu', 
    x_shift=0, 
    y_shift=0,
    batch_norm=True, 
    laplace_padding=None,
    input_regularizer='LaplaceL2norm',
    padding=False,
    final_nonlin=True,
    independent_bn_bias=True,
    momentum=0.7
)

shifter_dict = dict(
    gamma_shifter=0,
    shift_layers=3,
    input_channels_shifter=2,
    hidden_channels_shifter=5,
)


readout_dict = dict(
    bias=True,
    init_mu_range=0.1,
    init_sigma=0.3,
    gamma_readout=1.0,
    gauss_type='full',
    grid_mean_predictor={
        'type': 'cortex',
        'input_dimensions': 2,
        'hidden_layers': 1,
        'hidden_features': 30,
        'final_tanh': True
    },
    share_features=False,
    share_grid=False,
    shared_match_ids=None,
    gamma_grid_dispersion=0.0,
)
factorised_3d_model = make_video_model(
    data_loaders,
    seed,
    core_dict=factorised_3D_core_dict,
    core_type='3D_factorised',
    readout_dict=readout_dict.copy(),
    readout_type='gaussian',               
    use_gru=False,
    gru_dict=None,
    use_shifter=True,
    shifter_dict=shifter_dict,
    shifter_type='MLP',
)

print(factorised_3d_model)

factorised_3d_model.load_state_dict(torch.load(modelfile))
factorised_3d_model.eval()

## Get model predictions to looming stimuli

In [ ]:
stims = ['020', '032', '050', '080', '110'] # looming stimuli with these l/v params

stimuli_dir = "retina-axon-model/Analyses/Looming/stimuli/" # change to directory with looming stimuli
data_dir = "/data/"

all_outputs = {}
for stim in stims:
    print(stim)
    output_final = {}
    for data_key, dataloader in data_loaders['test'].items():
        print(data_key)
        
        stimulus_video = np.load(stimuli_dir + f"stim{stim}.npy").astype(np.float32)
        H, W, T = stimulus_video.shape  # (36, 64, t)
        
        pad_len = 50
        white_pad = np.full((H, W, pad_len), 255, dtype=stimulus_video.dtype)

        stimulus_video = np.concatenate((white_pad, stimulus_video), axis=-1)
        n = stimulus_video.shape[-1]  # padded length
        t = stimulus_video.shape[2] 
        
        # normalize looming video with input statistics
        statistics_folder = f"{data_dir}/{data_key}/meta/statistics/"
        signal = "videos"
        mean_videos = np.load(statistics_folder+signal+"/all"+"/mean.npy")[..., :n]
        std_videos = np.load(statistics_folder+signal+"/all"+"/std.npy")[..., :n]
        if mean_videos.shape[-1] < n:
            mean_videos = np.pad(mean_videos, pad_width=((0, 0), (0, 0), (0, n-mean_videos.shape[-1])), mode='edge')
            std_videos = np.pad(std_videos, pad_width=((0, 0), (0, 0), (0, n-std_videos.shape[-1])), mode='edge')

        stimulus_video = (stimulus_video - mean_videos) / std_videos
        
        stimulus_video = stimulus_video[None, :, :, :] 
        
        behavior_zeros = np.zeros((2, H, W, n), dtype=np.float32)
        stimulus_video = np.concatenate((stimulus_video, behavior_zeros), axis=0) 
        stimulus_video = np.transpose(stimulus_video, (0, 3, 1, 2))             

        stimulus_video = torch.from_numpy(stimulus_video).unsqueeze(0)             # shape (1, 3, n, 36, 64)
         
        # we set behavior and pupil_center to zeros
        behavior = torch.zeros((1, 2, n), dtype=torch.float32)
        pupil_center = torch.zeros((1, 2, n), dtype=torch.float32)

        output = {}
        for data_key2, dataloader in data_loaders['test'].items():
            with torch.no_grad():
                with device_state(factorised_3d_model, device):
                    out = (
                        factorised_3d_model(
                            stimulus_video.to(device), 
                            data_key=data_key2, 
                            pupil_center=pupil_center.to(device), 
                            behavior=behavior.to(device)
                        ) 
                        .detach()
                        .cpu()[:, -t :, :] 
                    )
                    output[data_key2] = np.squeeze(out.permute(0, 2, 1).numpy())
        output_final[data_key] = output[data_key]
    all_outputs[stim] = output_final